### Cell 08.01 — recover the frozen prot_03 QTL directly from the empirical-results workbook

In [ ]:
# Cell 08.01
# Load the final all-trait empirical QTL results and recover prot_03.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


QTL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)


print("QTL FILE EXISTS:", QTL_FILE.exists())


xls = pd.ExcelFile(QTL_FILE)

print("Sheets:")
print(xls.sheet_names)


# Find the sheet containing trait-level QTL results.
qtl_results = None
qtl_sheet_used = None

for sheet in xls.sheet_names:

    tmp = pd.read_excel(
        QTL_FILE,
        sheet_name=sheet
    )

    cols_lower = {
        str(c).lower()
        for c in tmp.columns
    }

    if (
        "trait" in cols_lower
        and
        any(
            x in cols_lower
            for x in [
                "peak_marker",
                "marker"
            ]
        )
    ):

        qtl_results = tmp.copy()
        qtl_sheet_used = sheet
        break


if qtl_results is None:
    raise ValueError(
        "Could not automatically locate the trait-level QTL results sheet."
    )


print()
print("QTL sheet used:", qtl_sheet_used)
print("Shape:", qtl_results.shape)
print("Columns:")
print(qtl_results.columns.tolist())

### Cell 08.02 — extract and freeze the prot_03 locus

In [ ]:
# Cell 08.02
# Extract the prot_03 peak without hardcoding effect direction.

prot_rows = (
    qtl_results
    .loc[
        qtl_results["trait"].astype(str)
        == "prot_03"
    ]
    .copy()
)


print("prot_03 rows found:", len(prot_rows))

display(prot_rows)


# Known independently anchored physical position
# from Notebook 05 / final physical localization.
SATT440_GNM6_BP = 49_904_048


PROT03_QTL = {
    "trait": "prot_03",
    "peak_marker": "Satt440",
    "structural_group": "pLG11",
    "physical_chr": "Gm20",
    "physical_assignment": "supported",

    "lod": 2.689816,
    "r2": 0.162185,
    "n": 70,
    "empirical_p": 0.070193,
    "significance_status": "suggestive_10pct",

    "peak_anchor_bp_gnm6": SATT440_GNM6_BP,
    "peak_anchor_mb_gnm6": SATT440_GNM6_BP / 1e6,

    "anchor_evidence": "exact",

    "physical_localization_status":
        "exact_peak_anchor_only_no_closed_bracket"
}


# Recover effect direction if available in workbook.
possible_effect_cols = [
    "effect_2_minus_0",
    "effect",
    "beta1"
]

for col in possible_effect_cols:

    if col in prot_rows.columns:

        PROT03_QTL[col] = (
            prot_rows.iloc[0][col]
        )


print("FROZEN prot_03 QTL")
print("=" * 90)

for key, value in PROT03_QTL.items():
    print(f"{key}: {value}")

### Cell 08.03 — reuse Wm82.gnm6 genes and inspect anchor-centered windows

In [ ]:
# Cell 08.03
# Extract genes around the exact Satt440 physical anchor.
#
# These windows are DESCRIPTIVE LOCAL CONTEXT ONLY.
# They are NOT statistical or physical QTL confidence intervals.

from urllib.parse import unquote


SOYBASE_REF_DIR = (
    PROJECT_ROOT
    / "reference"
    / "soybase"
)


GNM6_GENE_GFF = (
    SOYBASE_REF_DIR
    / "glyma.Wm82.gnm6.ann1.PKSW.gene_models_main.gff3.gz"
)


gff_columns = [
    "seqid",
    "source",
    "feature_type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes"
]


gff = pd.read_csv(
    GNM6_GENE_GFF,
    sep="\t",
    comment="#",
    names=gff_columns
)


genes = (
    gff
    .loc[
        gff["feature_type"] == "gene"
    ]
    .copy()
)


genes["start"] = pd.to_numeric(
    genes["start"],
    errors="coerce"
)

genes["end"] = pd.to_numeric(
    genes["end"],
    errors="coerce"
)


def parse_attrs(text):

    result = {}

    if pd.isna(text):
        return result

    for item in str(text).split(";"):

        if "=" in item:

            key, value = item.split(
                "=",
                1
            )

            result[key] = value

    return result


genes["attribute_dict"] = (
    genes["attributes"]
    .map(parse_attrs)
)


for field in [
    "Name",
    "Note",
    "Ontology_term",
    "Dbxref",
    "ancestorIdentifier"
]:

    genes[field] = (
        genes["attribute_dict"]
        .map(
            lambda d, f=field:
                d.get(f, np.nan)
        )
    )


genes["Note_decoded"] = (
    genes["Note"]
    .fillna("")
    .map(unquote)
)


genes["primary_annotation"] = (
    genes["Note_decoded"]
    .map(
        lambda x:
            str(x).split(";")[0].strip()
    )
)


gm20_genes = (
    genes
    .loc[
        genes["seqid"]
        == "glyma.Wm82.gnm6.Gm20"
    ]
    .copy()
)


ANCHOR_BP = SATT440_GNM6_BP


def genes_in_anchor_window(window_mb):

    window_bp = int(
        window_mb * 1_000_000
    )

    subset = (
        gm20_genes
        .loc[
            (gm20_genes["end"] >= ANCHOR_BP - window_bp)
            &
            (gm20_genes["start"] <= ANCHOR_BP + window_bp)
        ]
        .copy()
    )

    subset["start_mb"] = (
        subset["start"] / 1e6
    )

    subset["end_mb"] = (
        subset["end"] / 1e6
    )

    subset["midpoint_mb"] = (
        subset["start_mb"]
        + subset["end_mb"]
    ) / 2

    subset[
        "distance_to_Satt440_mb"
    ] = (
        subset["midpoint_mb"]
        - SATT440_GNM6_BP / 1e6
    ).abs()

    return (
        subset
        .sort_values(
            "distance_to_Satt440_mb"
        )
        .reset_index(drop=True)
    )


prot03_500kb = genes_in_anchor_window(
    0.5
)

prot03_1mb = genes_in_anchor_window(
    1.0
)

prot03_2mb = genes_in_anchor_window(
    2.0
)


print(
    "Genes within ±0.5 Mb:",
    len(prot03_500kb)
)

print(
    "Genes within ±1 Mb:",
    len(prot03_1mb)
)

print(
    "Genes within ±2 Mb:",
    len(prot03_2mb)
)


display(
    prot03_1mb[
        [
            "Name",
            "start_mb",
            "end_mb",
            "distance_to_Satt440_mb",
            "primary_annotation",
            "ancestorIdentifier"
        ]
    ]
)

### Cell 08.04 — save the anchor-centered checkpoint

In [ ]:
# Cell 08.04
# Export descriptive local genomic context for prot_03.

PROT03_CONTEXT_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_prot03_gm20_anchor_context.xlsx"
)


prot03_summary = pd.DataFrame(
    [
        {
            "trait": "prot_03",
            "peak_marker": "Satt440",
            "physical_chr": "Gm20",

            "lod": PROT03_QTL["lod"],
            "r2": PROT03_QTL["r2"],
            "n": PROT03_QTL["n"],
            "empirical_p": PROT03_QTL["empirical_p"],

            "significance_status":
                PROT03_QTL["significance_status"],

            "Satt440_bp_gnm6":
                SATT440_GNM6_BP,

            "anchor_evidence":
                "exact",

            "closed_physical_bracket":
                False,

            "genes_plus_minus_500kb":
                len(prot03_500kb),

            "genes_plus_minus_1Mb":
                len(prot03_1mb),

            "genes_plus_minus_2Mb":
                len(prot03_2mb),

            "interpretation":
                (
                    "Anchor-centered windows are descriptive "
                    "genomic context only and are not QTL "
                    "confidence intervals."
                )
        }
    ]
)


with pd.ExcelWriter(
    PROT03_CONTEXT_FILE,
    engine="openpyxl"
) as writer:

    prot03_summary.to_excel(
        writer,
        sheet_name="locus_summary",
        index=False
    )

    prot03_500kb.to_excel(
        writer,
        sheet_name="genes_500kb",
        index=False
    )

    prot03_1mb.to_excel(
        writer,
        sheet_name="genes_1Mb",
        index=False
    )

    prot03_2mb.to_excel(
        writer,
        sheet_name="genes_2Mb",
        index=False
    )


print("Saved:")
print(PROT03_CONTEXT_FILE)

### Cell 08.05 — define explicit seed-protein functional categories

In [ ]:
# Cell 08.05
# Conservative annotation-based screen for genes plausibly related to
# seed protein concentration, amino-acid metabolism, nitrogen metabolism,
# protein turnover, translation, and seed development.
#
# This is a functional screen only.
# It does NOT imply causal association with prot_03.

protein_candidate_rules = {

    "amino_acid_metabolism": [
        "cysteine synthase",
        "seryl-trna",
        "serine--trna",
        "amino acid",
        "amino-acid",
        "asparagine",
        "glutamine",
        "glutamate",
        "methionine",
        "lysine",
        "threonine aldolase"
    ],

    "nitrogen_metabolism": [
        "nitrogen",
        "carbon-nitrogen",
        "nitrate",
        "ammonium"
    ],

    "protein_turnover": [
        "protease",
        "proteasome",
        "peptidase",
        "ubiquitin",
        "protein degradation"
    ],

    "protein_translation": [
        "ribosomal protein",
        "trna synthetase",
        "trna ligase",
        "translation factor"
    ],

    "peptide_transport": [
        "peptide transporter"
    ],

    "seed_storage_or_seed_development": [
        "seed storage",
        "storage protein",
        "glycinin",
        "conglycinin",
        "legumin",
        "seed maturation",
        "embryo",
        "cotyledon"
    ],

    "carbon_lipid_partitioning": [
        "acyl-coa-binding",
        "acyl-coa binding",
        "lipoxygenase",
        "fatty acid",
        "lipid transfer"
    ],

    "transcriptional_regulation": [
        "transcription factor",
        "nuclear transcription factor",
        "bzip",
        "nf-y",
        "jumonji"
    ]
}


def classify_protein_candidate(annotation):

    text = str(annotation).lower()

    hits = []

    for category, terms in protein_candidate_rules.items():

        if any(term in text for term in terms):
            hits.append(category)

    return ";".join(hits)


prot03_1mb[
    "protein_candidate_categories"
] = (
    prot03_1mb[
        "primary_annotation"
    ]
    .map(classify_protein_candidate)
)


prot03_functional_screen = (
    prot03_1mb
    .loc[
        prot03_1mb[
            "protein_candidate_categories"
        ] != ""
    ]
    .copy()
    .sort_values(
        "distance_to_Satt440_mb"
    )
    .reset_index(drop=True)
)


print(
    "FUNCTIONAL CANDIDATES WITHIN ±1 Mb:",
    len(prot03_functional_screen)
)


display(
    prot03_functional_screen[
        [
            "Name",
            "start_mb",
            "distance_to_Satt440_mb",
            "protein_candidate_categories",
            "primary_annotation"
        ]
    ]
)

### Cell 08.06 — make a stricter metabolic/protein shortlist
* This excludes generic transcription factors unless they also hit another protein-related category.

In [ ]:
# Cell 08.06
# Build a stricter shortlist emphasizing direct biochemical relevance
# to protein concentration rather than generic regulatory genes.

direct_categories = [
    "amino_acid_metabolism",
    "nitrogen_metabolism",
    "protein_turnover",
    "protein_translation",
    "peptide_transport",
    "seed_storage_or_seed_development",
    "carbon_lipid_partitioning"
]


def has_direct_protein_category(categories):

    cats = str(categories).split(";")

    return any(
        cat in direct_categories
        for cat in cats
    )


prot03_direct_candidates = (
    prot03_functional_screen
    .loc[
        prot03_functional_screen[
            "protein_candidate_categories"
        ]
        .map(has_direct_protein_category)
    ]
    .copy()
    .sort_values(
        "distance_to_Satt440_mb"
    )
    .reset_index(drop=True)
)


print(
    "DIRECT FUNCTIONAL SHORTLIST:",
    len(prot03_direct_candidates)
)


display(
    prot03_direct_candidates[
        [
            "Name",
            "start_mb",
            "distance_to_Satt440_mb",
            "protein_candidate_categories",
            "primary_annotation",
            "ancestorIdentifier"
        ]
    ]
)

### Cell 08.07 — explicitly inspect the strongest local functional candidates

In [ ]:
# Cell 08.07
# Extract a focused set of biologically interpretable genes observed
# in the Satt440 ±1 Mb context.

focus_genes = [
    "Glyma.20G235500",  # acyl-CoA-binding protein
    "Glyma.20G236500",  # 26S protease regulatory subunit
    "Glyma.20G233300",  # seryl-tRNA synthetase
    "Glyma.20G229000",  # cysteine synthase D1
    "Glyma.20G228900",  # cysteine synthase D1
    "Glyma.20G243067",  # carbon-nitrogen hydrolase
    "Glyma.20G247400",  # peptide transporter
]


prot03_focus = (
    prot03_1mb
    .loc[
        prot03_1mb[
            "Name"
        ].isin(focus_genes)
    ]
    .copy()
    .sort_values(
        "distance_to_Satt440_mb"
    )
    .reset_index(drop=True)
)


prot03_focus[
    "evidence_status"
] = (
    "annotation_based_candidate_only"
)


display(
    prot03_focus[
        [
            "Name",
            "start_mb",
            "distance_to_Satt440_mb",
            "primary_annotation",
            "ancestorIdentifier",
            "evidence_status"
        ]
    ]
)

### Cell 08.08 — save the functional-screen checkpoint

In [ ]:
# Cell 08.08
# Save the annotation-based prot_03 candidate screen.

PROT03_SCREEN_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_prot03_gm20_functional_screen.xlsx"
)


with pd.ExcelWriter(
    PROT03_SCREEN_FILE,
    engine="openpyxl"
) as writer:

    prot03_summary.to_excel(
        writer,
        sheet_name="locus_summary",
        index=False
    )

    prot03_focus.to_excel(
        writer,
        sheet_name="focused_candidates",
        index=False
    )

    prot03_direct_candidates.to_excel(
        writer,
        sheet_name="direct_functional",
        index=False
    )

    prot03_functional_screen.to_excel(
        writer,
        sheet_name="all_functional",
        index=False
    )

    prot03_500kb.to_excel(
        writer,
        sheet_name="genes_500kb",
        index=False
    )

    prot03_1mb.to_excel(
        writer,
        sheet_name="genes_1Mb",
        index=False
    )


print("Saved:")
print(PROT03_SCREEN_FILE)

### Cell 08.09 — remove the obvious substring-driven false positives

In [ ]:
# Cell 08.09
# Curate the prot_03 shortlist by removing known keyword artifacts.
#
# This does NOT exclude genes because they are biologically impossible;
# it removes candidates whose inclusion arose from misleading substring matches.

prot03_curated = (
    prot03_direct_candidates
    .copy()
)


exclude_reason = {
    "Glyma.20G227500":
        "false_positive_keyword: histone-lysine methyltransferase; "
        "not direct amino-acid metabolism",

    "Glyma.20G242500":
        "false_positive_keyword: 'leguminosin' matched 'legumin'; "
        "not established seed storage protein",

    "Glyma.20G242600":
        "false_positive_keyword: 'leguminosin' matched 'legumin'; "
        "not established seed storage protein",
}


prot03_curated[
    "exclude_reason"
] = (
    prot03_curated["Name"]
    .map(exclude_reason)
    .fillna("")
)


prot03_curated[
    "retain_after_manual_review"
] = (
    prot03_curated[
        "exclude_reason"
    ] == ""
)


print(
    "Candidates before curation:",
    len(prot03_curated)
)

print(
    "Retained after curation:",
    int(
        prot03_curated[
            "retain_after_manual_review"
        ].sum()
    )
)

print(
    "Excluded:",
    int(
        (~prot03_curated[
            "retain_after_manual_review"
        ]).sum()
    )
)


display(
    prot03_curated.loc[
        ~prot03_curated[
            "retain_after_manual_review"
        ],
        [
            "Name",
            "primary_annotation",
            "exclude_reason"
        ]
    ]
)

### Cell 08.10 — create an evidence-aware final shortlist

In [ ]:
# Cell 08.10
# Evidence-aware shortlist.
#
# Ranking distinguishes:
# 1. independent annotation support,
# 2. plausible protein-metabolism mechanism,
# 3. physical proximity to Satt440.
#
# Proximity alone is NOT treated as causal evidence.

prot03_final = (
    prot03_curated
    .loc[
        prot03_curated[
            "retain_after_manual_review"
        ]
    ]
    .copy()
)


prot03_final[
    "independent_functional_support"
] = False

prot03_final[
    "evidence_note"
] = ""


# SoyBase independently supports cysteine synthase D1 / OAS-TL7
mask = (
    prot03_final["Name"]
    == "Glyma.20G228900"
)

prot03_final.loc[
    mask,
    "independent_functional_support"
] = True

prot03_final.loc[
    mask,
    "evidence_note"
] = (
    "SoyBase independently annotates this gene as cysteine "
    "synthase D1/OAS-TL7 with cysteine and amino-acid "
    "biosynthetic functions."
)


# Glyma.20G235500 independently identified as soybean ACBP
mask = (
    prot03_final["Name"]
    == "Glyma.20G235500"
)

prot03_final.loc[
    mask,
    "independent_functional_support"
] = True

prot03_final.loc[
    mask,
    "evidence_note"
] = (
    "Independently classified as a soybean acyl-CoA-binding "
    "protein; relevant primarily to lipid/carbon metabolism."
)


def assign_prot03_class(row):

    gene = row["Name"]

    if gene == "Glyma.20G228900":
        return "Tier_1A_amino_acid_biosynthesis_supported"

    if gene == "Glyma.20G229000":
        return "Tier_1B_amino_acid_biosynthesis_candidate"

    if gene in [
        "Glyma.20G236500",
        "Glyma.20G233300",
        "Glyma.20G243067",
        "Glyma.20G247400",
    ]:
        return "Tier_2_protein_or_nitrogen_metabolism"

    if gene == "Glyma.20G235500":
        return "Tier_2_carbon_lipid_partitioning"

    return "Tier_3_broad_metabolic_or_protein_function"


prot03_final[
    "evidence_class"
] = (
    prot03_final
    .apply(
        assign_prot03_class,
        axis=1
    )
)


tier_order = {
    "Tier_1A_amino_acid_biosynthesis_supported": 1,
    "Tier_1B_amino_acid_biosynthesis_candidate": 2,
    "Tier_2_protein_or_nitrogen_metabolism": 3,
    "Tier_2_carbon_lipid_partitioning": 4,
    "Tier_3_broad_metabolic_or_protein_function": 5,
}


prot03_final[
    "evidence_rank"
] = (
    prot03_final[
        "evidence_class"
    ]
    .map(tier_order)
)


prot03_final = (
    prot03_final
    .sort_values(
        [
            "evidence_rank",
            "distance_to_Satt440_mb"
        ]
    )
    .reset_index(drop=True)
)


display(
    prot03_final[
        [
            "Name",
            "start_mb",
            "distance_to_Satt440_mb",
            "evidence_class",
            "independent_functional_support",
            "primary_annotation",
            "evidence_note"
        ]
    ]
)

### Cell 08.11 — record the historical Satt440 seed-protein context
* This is worth preserving separately from candidate-gene evidence.

In [ ]:
# Cell 08.11
# Literature/context evidence at the marker level.
#
# Historical colocalization is supportive context only.
# It is NOT evidence that the same causal allele or gene is involved.

prot03_marker_context = pd.DataFrame(
    [
        {
            "marker": "Satt440",
            "chromosome": "Gm20",
            "current_trait": "prot_03",
            "current_lod": 2.689816,
            "current_empirical_p": 0.070193,
            "current_status": "suggestive_10pct",

            "published_trait":
                "seed glycinin plus beta-conglycinin",

            "published_marker_context":
                "Satt440-Satt102",

            "historical_support":
                "yes",

            "interpretation":
                (
                    "Independent soybean population previously "
                    "showed a seed-protein-composition QTL associated "
                    "with the Satt440 region. This supports regional "
                    "biological relevance but does not establish "
                    "identity of the causal locus."
                )
        }
    ]
)


display(prot03_marker_context)

### Cell 08.12 — final export and freeze Notebook 08

In [ ]:
# Cell 08.12
# Final prot_03 evidence summary.

PROT03_FINAL_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_prot03_gm20_final_evidence_summary.xlsx"
)


prot03_final_summary = pd.DataFrame(
    [
        {
            "trait": "prot_03",
            "peak_marker": "Satt440",
            "physical_chr": "Gm20",

            "peak_anchor_bp_gnm6":
                SATT440_GNM6_BP,

            "anchor_evidence":
                "exact",

            "lod":
                2.689816,

            "r2":
                0.162185,

            "empirical_p":
                0.070193,

            "genomewide_status":
                "suggestive_10pct",

            "closed_physical_qtl_bracket":
                False,

            "genes_within_1Mb":
                len(prot03_1mb),

            "functional_candidates_initial":
                len(prot03_functional_screen),

            "direct_candidates_initial":
                len(prot03_direct_candidates),

            "curated_candidates":
                len(prot03_final),

            "historical_Satt440_seed_protein_context":
                True,

            "important_caveat":
                (
                    "The ±1 Mb window is descriptive genomic context "
                    "around an exact marker anchor, not a QTL confidence "
                    "interval. Historical Satt440 colocalization does not "
                    "demonstrate the same causal variant."
                )
        }
    ]
)


with pd.ExcelWriter(
    PROT03_FINAL_FILE,
    engine="openpyxl"
) as writer:

    prot03_final_summary.to_excel(
        writer,
        sheet_name="locus_summary",
        index=False
    )

    prot03_marker_context.to_excel(
        writer,
        sheet_name="historical_context",
        index=False
    )

    prot03_final.to_excel(
        writer,
        sheet_name="curated_candidates",
        index=False
    )

    prot03_curated.to_excel(
        writer,
        sheet_name="curation_audit",
        index=False
    )

    prot03_1mb.to_excel(
        writer,
        sheet_name="all_genes_1Mb",
        index=False
    )


print("FINAL prot_03 EXPORT")
print("=" * 90)
print(PROT03_FINAL_FILE)
print()
print("Notebook 08 can now be frozen.")